In [1]:
# workaround for incredibly stupid vscode on snap bug
from os import environ
krb5ccname = environ["KRB5CCNAME"]
if "hostfs" in krb5ccname:
    krb5ccname = krb5ccname.replace("/var/lib/snapd/hostfs", "")
    environ["KRB5CCNAME"] = krb5ccname

# add gsl includes to root
environ["ROOT_INCLUDE_PATH"] = environ["ROOT_INCLUDE_PATH"] + ":" + environ["GSL_ROOT_DIR"] + "/include"

In [2]:
import ROOT
from analysis_framework import Dataset
from ObjectSelectionHelper import ObjectSelectionHelper

OBJ: TStyle	ildStyle	ILD Style : 0 at: 0x7e65510
OBJ: TStyle	ildStyle	ILD Style : 0 at: 0x7f121c0


In [3]:
# CLD
# x_angle = 0.030 # rad
# ILD
x_angle = 0.014 # rad
n_threads = 6
# prod = False
prod = True
no_rvec = True
write_outputs = False
# write_outputs = True
# plot_dir_postfix = "-new-cuts"
dataset_path = "data/datasets/selected-objects/test.json"
# output_path = "root://eosuser.cern.ch//eos/user/l/lreichen/TGC2/data/reweighted/test"
# output_meta_path = "data/datasets/reweighted"
# output_meta = f"{output_meta_path}/test.json"
# checked_output_meta = f"{output_meta_path}/checked-test.json"
# output_collections = [
#     "true_lep_lvec", "true_nu_lvec", "true_quark1_lvec", "true_quark2_lvec",
#     "iso_lep_lvec", "nu_lvec", "R2Jet_sel1_lvec", "R2Jet_sel2_lvec",
#     ]
# true lvecs do not exist in every df so cannot be explicitly requested...
# urgh but empty snapshots are also not allowed
# output_collections = r"(true_\w+_lvec)|(iso_lep_lvec)|(nu_lvec)|(R2Jet_sel1_lvec)|(R2Jet_sel2_lvec)|(\w*sqme\w*)|(weight\w*)"
# plot_dir = f"plots/pre-selection/test{plot_dir_postfix}"
if prod:
    # dataset_path = "data/datasets/miniDSTs/processed-no-exc-higgs.json"
    # dataset_path = "data/datasets/miniDSTs/processed-no-exc-higgs-min-aa-min-higgs.json"
    # dataset_path = "data/datasets/miniDSTs/min-higgs.json"
    dataset_path = "data/datasets/selected-objects/signal-only.json"
    # output_path = "root://eospublic.cern.ch//eos/experiment/clicdp/data/user/l/lreichen/snapshots3/min-higgs-d"
    # output_path = "root://eosuser.cern.ch//eos/user/l/lreichen/TGC2/data/reweighted/signal-only"
    # output_meta_path = "data/datasets/reweighted"
    # output_meta = f"{output_meta_path}/signal-only.json"
    # checked_output_meta = f"{output_meta_path}/checked-signal-only.json"
    # plot_dir = "plots/pre-selection/full"
    # plot_dir = f"plots/pre-selection/min-higgs{plot_dir_postfix}"


In [4]:
# ROOT.EnableImplicitMT(n_threads)
# environ["OMP_NUM_THREADS"] = "6"

In [5]:
dataset = Dataset.from_json(dataset_path)

In [6]:
analysis = ObjectSelectionHelper(dataset)

OBJ: TStyle	ildStyle	ILD Style : 0 at: 0xb5c1a70


In [7]:
analysis.init_categories()
# check if we missed any processes
print(analysis.is_complete_categorisation())
signal_category = ["4f_sw_sl_signal"]

True


In [8]:
ROOT.gInterpreter.Declare("#include \"kinfit.h\"")
ROOT.gSystem.Load("libMarlinKinfit.so")
# E_err = 4.4
# E_err = 3.5
# Theta_err = 0.045
# Phi_err = 0.04
# values for with BIB
# E_err = [0.85, 5.3, 3.5, 3.7]
# Theta_err = [3.3e-5, 0.71, 0.045, 0.035]
# Phi_err = [8.4e-5, 0.52, 0.039, 0.032]
# values for clean
E_err = [0.85, 5.0, 3.2, 3.4]
Theta_err = [3.3e-5, 0.70, 0.040, 0.030]
Phi_err = [8.4e-5, 0.53, 0.039, 0.033]
E_cms = 250.
m_W = 80.419
width_W = 2.049
fitter = ROOT.enuWFit(E_err, Theta_err, Phi_err, x_angle, E_cms, m_W, width_W)

In [9]:
reco_columns = ["iso_lep_lvec", "nu_lvec", "R2Jet_sel1_lvec", "R2Jet_sel2_lvec"]
# reco_columns = ["iso_lep_lvec", "clean_nu_lvec", "clean_R2Jet_sel1_lvec", "clean_R2Jet_sel2_lvec"]

In [10]:
analysis.Define("fitres", fitter, reco_columns)
analysis.Define("prob", "fitres.prob")
analysis.Define("chi2", "fitres.chi2")
analysis.Define("error", "fitres.error")
analysis.Define("postfit_iso_lep_lvec", "fitres.obj1")
analysis.Define("postfit_nu_lvec", "fitres.obj2")
analysis.Define("postfit_R2Jet1_lvec", "fitres.obj3")
analysis.Define("postfit_R2Jet2_lvec", "fitres.obj4")

In [11]:
analysis.define_deltas("postfit_iso_lep", "postfit_iso_lep_lvec", "true_lep_lvec", categories=signal_category)
analysis.define_deltas("postfit_nu", "postfit_nu_lvec", "true_nu_lvec", categories=signal_category)
analysis.define_deltas("postfit_R2Jet1", "postfit_R2Jet1_lvec", "true_quark1_lvec", categories=signal_category)
analysis.define_deltas("postfit_R2Jet2", "postfit_R2Jet2_lvec", "true_quark2_lvec", categories=signal_category)

analysis.define_deltas("iso_lep", "iso_lep_lvec", "true_lep_lvec", categories=signal_category)
analysis.define_deltas("nu", "nu_lvec", "true_nu_lvec", categories=signal_category)
analysis.define_deltas("R2Jet1", "R2Jet1_lvec", "true_quark1_lvec", categories=signal_category)
analysis.define_deltas("R2Jet2", "R2Jet2_lvec", "true_quark2_lvec", categories=signal_category)
# analysis.define_deltas("nu", "clean_nu_lvec", "true_nu_lvec", categories=signal_category)
# analysis.define_deltas("R2Jet1", "clean_R2Jet1_lvec", "true_quark1_lvec", categories=signal_category)
# analysis.define_deltas("R2Jet2", "clean_R2Jet2_lvec", "true_quark2_lvec", categories=signal_category)

In [12]:
analysis.book_histogram_1D("prob", "prob", ("", "", 100, 0., 1.))
analysis.book_histogram_1D("chi2", "chi2", ("", "", 100, 0., 50.))
analysis.book_histogram_1D("error", "error", ("", "", 20, -10., 10.))

In [13]:
analysis.add_filter("prob > 0.01", "prob > 0.01")

In [14]:
for prefix in ["", "postfit_"]:
    analysis.book_histogram_1D(f"{prefix}iso_lep_delta_P", f"{prefix}iso_lep_delta_P", ("", "", 150, -5., 5.,), categories=signal_category)
    analysis.book_histogram_1D(f"{prefix}nu_delta_P", f"{prefix}nu_delta_P", ("", "", 150, -25., 25.,), categories=signal_category)
    analysis.book_histogram_1D(f"{prefix}R2Jet1_delta_P", f"{prefix}R2Jet1_delta_P", ("", "", 150, -15., 15.,), categories=signal_category)
    analysis.book_histogram_1D(f"{prefix}R2Jet2_delta_P", f"{prefix}R2Jet2_delta_P", ("", "", 150, -15., 15.,), categories=signal_category)

    analysis.book_histogram_1D(f"{prefix}iso_lep_delta_theta", f"{prefix}iso_lep_delta_theta", ("", "", 150, -0.0005, 0.0005,), categories=signal_category)
    analysis.book_histogram_1D(f"{prefix}nu_delta_theta", f"{prefix}nu_delta_theta", ("", "", 150, -0.2, 0.2,), categories=signal_category)
    analysis.book_histogram_1D(f"{prefix}R2Jet1_delta_theta", f"{prefix}R2Jet1_delta_theta", ("", "", 150, -0.15, 0.15,), categories=signal_category)
    analysis.book_histogram_1D(f"{prefix}R2Jet2_delta_theta", f"{prefix}R2Jet2_delta_theta", ("", "", 150, -0.15, 0.15,), categories=signal_category)

    analysis.book_histogram_1D(f"{prefix}iso_lep_delta_phi", f"{prefix}iso_lep_delta_phi", ("", "", 150, -0.0005, 0.0005,), categories=signal_category)
    analysis.book_histogram_1D(f"{prefix}nu_delta_phi", f"{prefix}nu_delta_phi", ("", "", 150, -0.2, 0.2,), categories=signal_category)
    analysis.book_histogram_1D(f"{prefix}R2Jet1_delta_phi", f"{prefix}R2Jet1_delta_phi", ("", "", 150, -0.15, 0.15,), categories=signal_category)
    analysis.book_histogram_1D(f"{prefix}R2Jet2_delta_phi", f"{prefix}R2Jet2_delta_phi", ("", "", 150, -0.15, 0.15,), categories=signal_category)

In [15]:
if write_outputs:
    analysis.book_snapshots("events", output_path, output_meta, output_collections, no_rvec=no_rvec)

In [16]:
analysis.book_reports()

In [17]:
%%time
analysis.run()

CPU times: user 50.1 s, sys: 304 ms, total: 50.4 s
Wall time: 55.4 s


In [18]:
analysis.print_reports()

         4f_sw_sl_signal               4f_sl_bkg
        10244400 (1e-03)           33239 (2e-02) All
         7113796 (1e-03)           15088 (3e-02) prob > 0.01
                    0.69                    0.45 efficiency



In [19]:
# if write_outputs:
    # analysis.check_snapshots("events", output_path, checked_output_meta)

In [20]:
analysis.draw_histogram("prob", categories=signal_category)
analysis.draw_histogram("chi2")
analysis.draw_histogram("error")

(<cppyy.gbl.THStack object at 0x13b3d7e0>,
 <cppyy.gbl.TCanvas object at 0xfcf0e10>)

In [21]:
analysis.compare_summed_histograms_unscaled(["iso_lep_delta_P", "postfit_iso_lep_delta_P"], category=signal_category[0])
analysis.compare_summed_histograms_unscaled(["nu_delta_P", "postfit_nu_delta_P"], category=signal_category[0])
analysis.compare_summed_histograms_unscaled(["R2Jet1_delta_P", "postfit_R2Jet1_delta_P"], category=signal_category[0])
analysis.compare_summed_histograms_unscaled(["R2Jet2_delta_P", "postfit_R2Jet2_delta_P"], category=signal_category[0])

In [22]:
analysis.compare_summed_histograms_unscaled(["iso_lep_delta_theta", "postfit_iso_lep_delta_theta"], category=signal_category[0])
analysis.compare_summed_histograms_unscaled(["nu_delta_theta", "postfit_nu_delta_theta"], category=signal_category[0])
analysis.compare_summed_histograms_unscaled(["R2Jet1_delta_theta", "postfit_R2Jet1_delta_theta"], category=signal_category[0])
analysis.compare_summed_histograms_unscaled(["R2Jet2_delta_theta", "postfit_R2Jet2_delta_theta"], category=signal_category[0])

In [23]:
analysis.compare_summed_histograms_unscaled(["iso_lep_delta_phi", "postfit_iso_lep_delta_phi"], category=signal_category[0])
analysis.compare_summed_histograms_unscaled(["nu_delta_phi", "postfit_nu_delta_phi"], category=signal_category[0])
analysis.compare_summed_histograms_unscaled(["R2Jet1_delta_phi", "postfit_R2Jet1_delta_phi"], category=signal_category[0])
analysis.compare_summed_histograms_unscaled(["R2Jet2_delta_phi", "postfit_R2Jet2_delta_phi"], category=signal_category[0])